# 14 · Residual CatBoost on top of the formula

`blend_v4` averages two independently trained models: 0.75 × formula `SPEC4` + 0.25 × CatBoost (`07`); 15-fold AUC 0.7212.

The structurally cleaner combination is **boosting from the formula**: CatBoost starts from the formula's log-odds (`baseline`) and only learns what the formula misses. `13` measured that this adds ~+0.0002 over the formula. This notebook:
1. builds out-of-fold predictions of **formula + residual CatBoost** on the fixed 15 folds (CatBoost `07` parameters, lr 0.02, depth 4; tree count = best mean-AUC iteration, as in `cv_score`);
2. searches blend weights over {formula, formula + residual CatBoost, CatBoost} on seed 0, judged on seeds 1–2 against `blend_v4`;
3. writes `blend_v5` only if it beats `blend_v4` by > 2 se on seeds 1–2.

In [1]:
import json, sys, glob, itertools
import numpy as np, pandas as pd
import catboost as cbm
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
sys.path.insert(0, '.')
import common as C, features as FE, formula as FM
import formula_steps as FS
L, y, lab, hold = FS.L, FS.y, FS.lab, FS.hold
PC = json.load(open('params_cats.json')); FX = PC['features']
LC = FE.add_cats(L); cats = [c for c in FX if not pd.api.types.is_numeric_dtype(LC[c])]
cb15 = C.cv_score('cats15___copies__ohm_10', FE.add_cats(FE.add_static(lab)), FX, 'cb', PC['params'])
f15 = FS.run4(FS.SPEC4, 0.1, seeds=(0, 1, 2))
FOLDS = C.folds(lab)
RP = PC['params'] | {'learning_rate': 0.02}
sig = lambda z: 1 / (1 + np.exp(-z))
print(f'formula SPEC4 {f15["auc_mean"]:.4f} | CatBoost {cb15["auc_mean"]:.4f}')

formula SPEC4 0.7206 | CatBoost 0.7143


## 1. Out-of-fold formula + residual CatBoost

In [2]:
fits = []
for s, k, tr, va in FOLDS:
    m = FM.Formula(FS.SPEC4, 0.1, '2era').fit(L.iloc[tr], y[tr])
    btr, bva = m.decision(L.iloc[tr]), m.decision(L.iloc[va])
    Xva = C.cb_X(LC.iloc[va][FX], cats)
    r = cbm.CatBoostClassifier(**RP, iterations=3000, early_stopping_rounds=200, use_best_model=False)
    r.fit(cbm.Pool(C.cb_X(LC.iloc[tr][FX], cats), y[tr], cat_features=cats, baseline=btr),
          eval_set=cbm.Pool(Xva, y[va], cat_features=cats, baseline=bva))
    fits.append((s, va, bva, Xva, r, np.array(r.get_evals_result()['validation']['AUC'])))
Lm = max(len(f[5]) for f in fits)
best_it = int(np.mean([np.pad(f[5], (0, Lm - len(f[5])), mode='edge') for f in fits], axis=0).argmax()) + 1
RES = np.full((3, len(y)), np.nan)
for s, va, bva, Xva, r, _ in fits:
    raw = r.predict(cbm.Pool(Xva, cat_features=cats), prediction_type='RawFormulaVal', ntree_end=min(best_it, r.tree_count_))
    RES[s, va] = sig(bva + raw)
chk = roc_auc_score(y[fits[0][1]], RES[0, fits[0][1]])
print(f'trees: {best_it} | fold-0 check: AUC from predictions {chk:.4f} vs eval curve {fits[0][5][min(best_it, len(fits[0][5])) - 1]:.4f}')
aucs = lambda P: np.array([roc_auc_score(y[va], P[s, va]) for s, k, tr, va in FOLDS])
M = {'formula': f15['oof_all'], 'formula + residual CB': RES, 'CatBoost': cb15['oof_all']}
d = aucs(RES) - aucs(M['formula'])
print(f'formula + residual CatBoost: 15-fold {aucs(RES).mean():.4f} | vs formula {d.mean():+.5f} ± {d.std()/np.sqrt(15):.5f} '
      f'(seeds 1-2 {d[5:].mean():+.5f} ± {d[5:].std()/np.sqrt(10):.5f})')
print('rank correlations (repeat 0):', {f'{a} / {b}': round(spearmanr(M[a][0], M[b][0])[0], 3) for a, b in itertools.combinations(M, 2)})

trees: 71 | fold-0 check: AUC from predictions 0.7267 vs eval curve 0.7267
formula + residual CatBoost: 15-fold 0.7208 | vs formula +0.00025 ± 0.00015 (seeds 1-2 +0.00025 ± 0.00014)
rank correlations (repeat 0): {'formula / formula + residual CB': np.float64(0.999), 'formula / CatBoost': np.float64(0.938), 'formula + residual CB / CatBoost': np.float64(0.943)}


**Reading:**
- **Formula + residual CatBoost (71 trees) gives 0.7208,** +0.00025 ± 0.00014 over the formula alone, matching `13`.
- **It barely changes the formula's ranking** (rank correlation 0.999), so it carries almost none of CatBoost's independent view (correlation 0.94).

## 2. Blend weights (seed 0) and judgment (seeds 1–2) vs `blend_v4`

In [3]:
v4 = 0.75 * M['formula'] + 0.25 * M['CatBoost']
steps = np.round(np.arange(0, 1.0001, 0.05), 2)
W = [np.array([a, b, round(1 - a - b, 2)]) for a in steps for b in steps if a + b <= 1 + 1e-9]
mix = lambda w: sum(wi * P for wi, P in zip(w, M.values()))
s0 = [aucs(mix(w))[:5].mean() for w in W]
wb = W[int(np.argmax(s0))]
base = aucs(v4)
cands = {'blend_v4 (0.75 formula + 0.25 CatBoost)': v4, 'formula alone': M['formula'], 'formula + residual CB alone': RES,
         '0.75 (formula + residual CB) + 0.25 CatBoost': 0.75 * RES + 0.25 * M['CatBoost'],
         f'best weights {dict(zip(M, wb))}': mix(wb)}
rows = {}
for n, P in cands.items():
    a = aucs(P); dd = a - base
    rows[n] = {'AUC (15)': a.mean(), 'Δ vs blend_v4': dd.mean(), 'se': dd.std() / np.sqrt(15), 'Δ seeds 1-2': dd[5:].mean(),
               'se (1-2)': dd[5:].std() / np.sqrt(10), 'top-decile': np.mean([C.top_decile_precision(y, P[s]) for s in range(3)])}
tb = pd.DataFrame(rows).T
display(tb.style.format({'AUC (15)': '{:.4f}', 'Δ vs blend_v4': '{:+.5f}', 'se': '{:.5f}', 'Δ seeds 1-2': '{:+.5f}', 'se (1-2)': '{:.5f}', 'top-decile': '{:.1%}'}))
cand = tb.drop(index=tb.index[0]).sort_values('Δ seeds 1-2', ascending=False)
pick = cand.index[0]
adopt = cand.loc[pick, 'Δ seeds 1-2'] > 2 * cand.loc[pick, 'se (1-2)']
print('best candidate:', pick, '| adopt:', adopt)

,AUC (15),Δ vs blend_v4,se,Δ seeds 1-2,se (1-2),top-decile
blend_v4 (0.75 formula + 0.25 CatBoost),0.7212,+0.00000,0.00000,+0.00000,0.00000,31.1%
formula alone,0.7206,-0.00067,0.00032,-0.00070,0.00042,31.1%
formula + residual CB alone,0.7208,-0.00042,0.00027,-0.00045,0.00039,31.0%
0.75 (formula + residual CB) + 0.25 CatBoost,0.7212,-0.00001,0.00013,-0.00000,0.00013,31.0%
"best weights {'formula': np.float64(0.4), 'formula + residual CB': np.float64(0.35), 'CatBoost': np.float64(0.25)}",0.7213,+0.00004,0.00006,+0.00004,0.00006,31.2%


best candidate: best weights {'formula': np.float64(0.4), 'formula + residual CB': np.float64(0.35), 'CatBoost': np.float64(0.25)} | adopt: False


In [4]:
if adopt:
    H = FM.prep(hold); HC = FE.add_cats(H)
    m = FM.Formula(FS.SPEC4, 0.1, '2era').fit(L, y)
    bl, bh = m.decision(L), m.decision(H)
    Xl, Xh = C.cb_X(LC[FX], cats), C.cb_X(HC[FX], cats)
    raw = np.mean([cbm.CatBoostClassifier(**(RP | {'random_seed': sd}), iterations=int(round(best_it * 1.2)))
                   .fit(cbm.Pool(Xl, y, cat_features=cats, baseline=bl))
                   .predict(cbm.Pool(Xh, cat_features=cats), prediction_type='RawFormulaVal') for sd in [42, 1, 2, 3, 4]], axis=0)
    comp = {'formula': m.predict(H), 'formula + residual CB': sig(bh + raw)}
    cbp = pd.read_csv(sorted(glob.glob('../submissions/catboost_cats5_*.csv'))[-1]); assert (cbp.POLICY_ID.values == hold.POLICY_ID.values).all()
    comp['CatBoost'] = cbp.CONVERT_IND_PRED.to_numpy()
    w = wb if pick.startswith('best') else (np.array([0, 0.75, 0.25]) if pick.startswith('0.75') else np.array([0, 1, 0]))
    p = sum(wi * comp[k] for wi, k in zip(w, M) if wi > 0)
    out = C.write_submission('blend_v5', hold, p)
    json.dump({'weights': dict(zip(M, map(float, w))), 'residual_trees': best_it, 'residual_params': {k: v for k, v in RP.items()},
               'cv_auc_15': float(tb.loc[pick, 'AUC (15)'])}, open('blend_v5.json', 'w'), indent=1, default=str)
    print(out.name, f'| {pd.Series(p).nunique()} distinct, mean {p.mean():.4f}, rank corr with blend_v4 inputs:',
          round(spearmanr(p, 0.75 * comp['formula'] + 0.25 * comp['CatBoost'])[0], 4))
else:
    print('no change: blend_v4 stays the final submission')

no change: blend_v4 stays the final submission


## Conclusions
- **No change: `blend_v4` (0.75 formula + 0.25 CatBoost) stays the final submission.**
- **Swapping in the residual model:** 0.75 × (formula + residual CatBoost) + 0.25 × CatBoost ties `blend_v4` exactly (Δ 0.0000 ± 0.0001 on seeds 1–2).
- **Best searched weights:** 0.40 formula + 0.35 residual + 0.25 CatBoost reaches 0.7213, only +0.00004 ± 0.00006, far from the 2·se bar.
- **Why it doesn't add:** the residual model and the standalone CatBoost capture the same small leftover signal (~+0.0002–0.0006). The standalone CatBoost brings the bigger benefit because its errors are more independent of the formula's.
- With this, the modeling search has converged at **15-fold AUC ≈ 0.721**.